# Overview — Tổng hợp Sensitivity Analysis 4 cụm

Bảng so sánh và tornado tổng hợp xuyên cụm. Mỗi cụm đã có notebook riêng (01–04).

> **Lưu ý đo lường:** |Δ| thô của mỗi cụm nằm trên *thang điểm khác nhau* (A: thang reputation; B: composite 0–100; C: trust cộng dồn; D: trust scaled 0–100), nên **không so sánh trực tiếp** được. Tornado tổng hợp dưới đây **chuẩn hoá theo từng cụm** (chia cho |Δ| lớn nhất trong cụm) → thể hiện *tầm quan trọng tương đối* của mỗi tham số *bên trong* cụm của nó.

Snapshot: `snap_20260613_185128` (cùng một snapshot cho cả 4 cụm).

In [ ]:
import sys
sys.path.insert(0, '..')
from lib import setup_thesis_style, save_figure, load_tornado
import pandas as pd
import matplotlib.pyplot as plt

setup_thesis_style()
SNAP_ID = "snap_20260613_185128"
OUTPUT_DIR = f"../output/{SNAP_ID}"

clusters = ["A", "B", "C", "D"]
tornados = {}
for cl in clusters:
    try:
        tornados[cl] = load_tornado(OUTPUT_DIR, cl)
    except FileNotFoundError:
        print(f"Cluster {cl}: tornado.csv not found")

In [ ]:
rows = []
for cl, df in tornados.items():
    sub = df.copy()
    sub["total"] = sub["delta_low"] + sub["delta_high"]
    mx = sub["total"].max()
    sub["rel"] = sub["total"] / mx if mx > 0 else 0.0
    sub["cluster"] = cl
    rows.append(sub)
combined = pd.concat(rows, ignore_index=True)
# Keep the top-4 relative params of each cluster for a readable chart.
top = (combined.sort_values("rel", ascending=False)
                .groupby("cluster", group_keys=False).head(4))
top = top.sort_values(["cluster", "rel"])

color_map = {"A": "#1f77b4", "B": "#ff7f0e", "C": "#2ca02c", "D": "#d62728"}
fig, ax = plt.subplots(figsize=(8, 9))
labels = top["param"] + "  [" + top["cluster"] + "]"
ax.barh(labels, top["rel"], color=[color_map[c] for c in top["cluster"]])
handles = [plt.Rectangle((0, 0), 1, 1, color=color_map[c]) for c in clusters if c in tornados]
ax.legend(handles, [f"Cluster {c}" for c in clusters if c in tornados], loc="lower right")
ax.set_xlabel("Tầm quan trọng tương đối (|Δ| chuẩn hoá theo cụm)")
ax.set_title("Overview — Top-4 tham số nhạy nhất mỗi cụm (chuẩn hoá nội cụm)")
ax.set_xlim(0, 1.05)
fig.tight_layout()
save_figure(fig, "overview_combined_tornado")
plt.show()

In [ ]:
names = {"A": "Scoring Core", "B": "Composite Blend",
         "C": "Quality+Propagation", "D": "Power Iteration"}
summary = []
for cl in clusters:
    if cl not in tornados:
        continue
    df = tornados[cl].copy()
    df["total"] = df["delta_low"] + df["delta_high"]
    df = df.sort_values("total", ascending=False)
    top1 = df.iloc[0]
    summary.append({
        "Cluster": cl,
        "Tên cụm": names[cl],
        "# params": len(df),
        "Tham số nhạy nhất": top1["param"],
        "|Δ| (thô)": round(top1["total"], 4),
    })
print(pd.DataFrame(summary).to_string(index=False))